In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# 公开 Baseline 分片 2

本入口已预填 `ROLE = 'public'`、`SHARD = 2` 和 `PREVIOUS_RUNTIME_STOPPED = True`。本次用户已确认四个旧 public runtime 全部终止断开；选择独立 GPU runtime，挂载原 Google Drive 后直接 Run all，继续原 `attempt01/shards/part2`。读取保存的 checkpoint、批次结果和调用记录，复用已完成计算，继续未完成工作，不新建结果轮次。PixelSeal/FrameIndex 共享计算保留在同片。不同 public 分片可并行；相同分片只运行一份。所有计算 worker 停止或暂停后，单独运行[唯一 CPU 汇总](https://colab.research.google.com/github/RICHAAARC/SC-SSTW/blob/main/notebooks/paper_finalize_v1.ipynb)。

[运行顺序、分片、恢复和时长说明](https://github.com/RICHAAARC/SC-SSTW/blob/main/docs/paper_execution_v1.md)


In [ ]:
SOURCE_REF = 'main'
SOURCE_DIRECTORY = None  # 可选可读源码，不要求版本摘要一致
PREVIOUS_RUNTIME_STOPPED = True  # 本次已确认旧 public runtime 全部终止断开
MAX_ITEMS = None  # 默认持续运行，不按源/pack数量自动暂停
MAX_HOURS = None  # 默认持续运行，不设置人为时长暂停；原科学资源上限保留
ROLE = 'public'  # 本入口已预填，无需修改
SHARD = 2  # 本入口已预填，无需修改


续跑原目录：`MyDrive/Video-WM_Result/baselines/public-native-frameindex-compact12-v1/attempt01/shards/part2`。保留原 checkpoint、已提交批次、旧失败和调用记录；只补未完成工作及已修复的基础设施额度故障，不重置科学分母或原始调用账。完整 CPU 画质记录直接复用，不追加 LPIPS。

源码默认加载当前 `main`，已包含模型重新加载和基础设施续跑修复：不再按模型加载次数或恢复次数人为停止；`MAX_ITEMS`、`MAX_HOURS` 均为 `None`。实际文件、数值或模型错误仍记录。旧 ACTIVE 占用记录可按已确认停止恢复，不覆盖仍在本机运行的同片进程。不承诺模型内部任意位置无损恢复；未保存的物理调用及额外恢复成本保留记录。


In [ ]:
from pathlib import Path
import urllib.request
bootstrap_path = 'runtime/wan/paper_suite_colab_v1.py'
if SOURCE_DIRECTORY:
    bootstrap = (Path(SOURCE_DIRECTORY) / bootstrap_path).read_text()
else:
    bootstrap = urllib.request.urlopen(f'https://raw.githubusercontent.com/RICHAAARC/SC-SSTW/{SOURCE_REF}/{bootstrap_path}', timeout=120).read().decode('utf-8')
entry = {'__name__': 'paper_suite_colab_entry'}
exec(compile(bootstrap, bootstrap_path, 'exec'), entry)
RUN = entry['run_parallel_colab'](source_directory=SOURCE_DIRECTORY, source_ref=SOURCE_REF, previous_runtime_stopped=PREVIOUS_RUNTIME_STOPPED, max_items=MAX_ITEMS, max_hours=MAX_HOURS, role=ROLE, shard=SHARD)


正式结果与失败包均保留在 `MyDrive/Video-WM_Result`；可恢复大数组放独立 `MyDrive/Video-WM_Cache`，暂停时不删除。只有完整结束的自有临时缓存可以释放。分片仅改变调度，不改变 DCT2＋CDC32、旧 H/L、固定源/视图/分母、阈值或科学调用预算。

入口返回、测量完成与科学结论分开；随机查询条件界不等于固定身份的总体 FPR 认证。公开自然视频与同 Wan 对照分表报告。


In [ ]:
print('入口状态:', RUN['status'])
print('本次记录:', RUN['invocation'])
print('各片返回:', [(row['stage'], row['result'].get('status')) for row in RUN['stages']])
entry['show_results']()
